In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate gradio

In [ ]:
import os
import requests
from google.colab import drive
from huggingface_hub import login
from openai import OpenAI
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
from IPython.display import Markdown, display, update_display


In [3]:
AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [4]:
audio_filename="/content/Creador de Godot ｜ Juan Linietsky ｜ Entrevistas Press Over.mp3"

In [5]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [6]:
openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

In [7]:
audio_file = open(audio_filename, "rb")
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

En este caso estamos con Juan Liniesky, uno de los fundadores de la Fundación Argentina de Videojuegos y además creador del motor Godot. ¿Cómo estás, Juan? Bien, ¿qué tal? ¿Cómo estás vos? Todo bien. Contanos un poco qué viniste a hacer a Núcleo. Vine a una charla sobre Godot, a contar un poquito sobre la versión nueva que va a salir dentro de poco. Godot es un engine abierto de código fuente, cualquiera se lo puede bajar, lo puede usar y sin ningún tipo de restricción, es casi como si lo hubieras hecho vos, que tenés las posibilidades de usarlo, así que veo que hay mucho interés por la versión nueva, así que armamos acá en este evento una charla muy linda, gracias a la gente de Image, pudimos hacerlo. Y bueno, eso. Buenísimo. En diciembre, me decís qué sale. Supuestamente a fin de diciembre va a salir la versión nueva, vamos a ver cómo viene, pero la idea es a fines de diciembre. Estamos viendo por alguna de las paredes de Image que habían panfletos que decían el Gran Linieski. Contan

In [13]:
system_message = "Eres un asistente que produce resumenes de entrevistas a partir de transcripciones,con resumen, puntos clave de preguntas, conclusiones, en formato Markdown"
user_prompt = f"A continuación se muestra un extracto de la entrevista a Juan Liniesky uno de los fundadores de Godot. Por favor, escriba las actas en formato Markdown, incluyendo un resumen de la entrevista; las preguntas y respuestas; las conclusiones; y las acciones a tomar con los responsables.\n{transcription}"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

In [14]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Eres un asistente que produce resumenes de entrevistas a partir de transcripciones,con resumen, puntos clave de preguntas, conclusiones, en formato Markdown<|eot_id|><|start_header_id|>user<|end_header_id|>

A continuación se muestra un extracto de la entrevista a Juan Liniesky uno de los fundadores de Godot. Por favor, escriba las actas en formato Markdown, incluyendo un resumen de la entrevista; las preguntas y respuestas; las conclusiones; y las acciones a tomar con los responsables.
En este caso estamos con Juan Liniesky, uno de los fundadores de la Fundación Argentina de Videojuegos y además creador del motor Godot. ¿Cómo estás, Juan? Bien, ¿qué tal? ¿Cómo estás vos? Todo bien. Contanos un poco qué viniste a hacer a Núcleo. Vine a una charla sobre Godot, a contar un poquito sobre la versión nueva que va a salir dentro de poco. Godot es un engine abierto de có

In [16]:
response = tokenizer.decode(outputs[0])

In [17]:
display(Markdown(response))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Eres un asistente que produce resumenes de entrevistas a partir de transcripciones,con resumen, puntos clave de preguntas, conclusiones, en formato Markdown<|eot_id|><|start_header_id|>user<|end_header_id|>

A continuación se muestra un extracto de la entrevista a Juan Liniesky uno de los fundadores de Godot. Por favor, escriba las actas en formato Markdown, incluyendo un resumen de la entrevista; las preguntas y respuestas; las conclusiones; y las acciones a tomar con los responsables.
En este caso estamos con Juan Liniesky, uno de los fundadores de la Fundación Argentina de Videojuegos y además creador del motor Godot. ¿Cómo estás, Juan? Bien, ¿qué tal? ¿Cómo estás vos? Todo bien. Contanos un poco qué viniste a hacer a Núcleo. Vine a una charla sobre Godot, a contar un poquito sobre la versión nueva que va a salir dentro de poco. Godot es un engine abierto de código fuente, cualquiera se lo puede bajar, lo puede usar y sin ningún tipo de restricción, es casi como si lo hubieras hecho vos, que tenés las posibilidades de usarlo, así que veo que hay mucho interés por la versión nueva, así que armamos acá en este evento una charla muy linda, gracias a la gente de Image, pudimos hacerlo. Y bueno, eso. Buenísimo. En diciembre, me decís qué sale. Supuestamente a fin de diciembre va a salir la versión nueva, vamos a ver cómo viene, pero la idea es a fines de diciembre. Estamos viendo por alguna de las paredes de Image que habían panfletos que decían el Gran Linieski. Contanos un poco cómo surgió ese panfleto. Yo no tuve nada que ver, soy inocente, no tengo idea cómo apareció eso ahí. Igual no he sorprendido que vos. Sí, sí, sí, nosotros veníamos caminando y veíamos tipo como si fuera casi como un houdini. No sé, no hago magia, no sé, no soy mago. La verdad que hay mucha recepción muy positiva para todo el trabajo que estoy haciendo, que quiero aclarar que no es algo que hago yo solo, lo hacen casi 400 personas que están contribuyendo al proyecto. Si vos ves por ejemplo gráficos de curvas de cantidad de gente que contribuye, cuando salió no había nadie, estaba yo solo escribiendo código hace 3 años, y va creciendo de forma exponencial, hay casi 400 personas que están contribuyendo con 30 o 40 que lo hacen casi como si fuera su trabajo, pese a que nadie les está pagando. Lo que es interesante de Godot es que, por la diferencia de otros motores, todo lo que uno hace por Godot lo puede usar para uno después, es como una especie de esfuerzo y recompensa equitativa entre todos, no es que una empresa se quede con el trabajo de los demás. La gente se motiva mucho y entra al proyecto y quiere contribuir y agregar las cosas que necesita. Bueno, y vos estás en la industria argentina de videojuegos de hace más de 20 años, ¿no? Sí, soy uno de los fundadores, fundador de Adobe y fundador de reuniones en Burger King de hace 20 años, pero sí. Hace poco estuvimos viendo una foto de hace mucho tiempo, y la verdad que se mantienen bastante bien todos, ¿eh? No, estábamos todos ahí en el grupo de WhatsApp diciendo, boludo, está mucho mierda. Pero bueno, la verdad que está bueno porque en ese entonces soñábamos con tener una industria acá y era un sueño que decía, esto no va a pasar nunca, y bueno, ahora aquí estamos. Tenemos la industria de videojuegos más importante de Latinoamérica y seguimos creciendo. Sí, es genial. Buenísimo, Juan. Muchísimas gracias. No, gracias a ustedes. Gracias, Juan.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Resumen de la entrevista**

Juan Liniersky, uno de los fundadores de la Fundación Argentina de Videojuegos y creador del motor Godot, se reunió con nosotros en Núcleo para hablar sobre la versión nueva de Godot que se lanzará a finales de diciembre. Godot es un engine abierto de código fuente que permite a cualquier persona descargarlo, usarlo y contribuir a su desarrollo sin restricciones. Juan destacó la comunidad que rodea a Godot, con más de 400 personas contribuyendo al proyecto, y la filosofía de recompensa equitativa que permite a los desarrolladores usar sus contribuciones para su propio beneficio.

**Preguntas y respuestas**

1. ¿Qué viniste a hacer a Núcleo?
* Vine a una charla sobre Godot, a contar un poquito sobre la versión nueva que va a salir dentro de poco.
2. ¿Qué es Godot?
* Godot es un engine abierto de código fuente, cualquiera se lo puede bajar, lo puede usar y sin ningún tipo de restricción.
3. ¿Qué sale la versión nueva de Godot?
* Supuestamente a fin de diciembre va a salir la versión nueva, vamos a ver cómo viene, pero la idea es a fines de diciembre.
4. ¿Cómo surgió el panfleto con el nombre "Gran Linieski"?
* Yo no tuve nada que ver, soy inocente, no tengo idea cómo apareció eso ahí.
5. ¿Cuántas personas contribuyen a Godot?
* Casi 400 personas que están contribuyendo al proyecto.
6. ¿Qué es lo que hace que la gente se motive a contribuir a Godot?
* Todo lo que uno hace por Godot lo puede usar para uno después, es como una especie de esfuerzo y recompensa equitativa entre todos.
7. ¿Cuántos años tienes en la industria argentina de videojuegos?
* Hace más de 20 años, soy uno de los fundadores de la Fundación Argentina de Videojuegos.

**Conclusión**

La entrevista con Juan Liniersky nos ha dado una visión clara de la filosofía y la comunidad que rodean a Godot. La versión nueva que se lanzará a finales de diciembre promete ser una actualización significativa del motor, y la comunidad que lo rodea está comprometida con su desarrollo y uso. Es importante destacar la filosofía de recompensa equitativa que permite a los desarrolladores usar sus contribuciones para su propio beneficio.

**Acciones a tomar con los responsables**

* Comunicar a la comunidad de Godot sobre la entrevista y la versión nueva que se lanzará a finales de diciembre.
* Invitar a Juan Liniersky a participar en futuras charlas y eventos relacionados con Godot.
* Explorar la posibilidad de colaborar con la Fundación Argentina de Videojuegos para apoyar el desarrollo de la industria de videojuegos en Argentina.<|eot_id|>